## Troubleshooting Local Transformers Translation Logic

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Verify CUDA and GPU availability
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0))

CUDA available: True
Device name: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [ ]:
# Translation model (No Language Left Behind | Meta)
model_name = 'facebook/nllb-200-distilled-600M'

# Load model vocab for Portuguese input
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang = 'por_Latn')

# Load model weights, use 16-bit FP to reduce VRAM usage by half, and move model to GPU
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name, dtype = torch.float16
).to("cuda")

Loading weights: 100%|██████████| 512/512 [00:01<00:00, 290.73it/s]


In [ ]:
# Example reviews
port_texts = ['O sanduíche estava ótimo!', 'Entrega muito rápida, produto excelente!', 'Não gostei da qualidade.']

# Convert strings to token ID tensors, pad short sentences, truncate input longer than 1024 tokens
inputs = tokenizer(texts, return_tensors = 'pt', padding = True, truncation = True).to('cuda')

In [ ]:
# Disable gradient calculation to reduce VRAM usage (.no_grad())
with torch.no_grad():
    # Translation text-generation loop
    generated = model.generate(
        # Unpacks input_ids and attention mask tensors
        **inputs,
        # Forces Beginning of Sentence token to English (target output language)
        forced_bos_token_id = tokenizer.convert_tokens_to_ids('eng_Latn'),
        # Constrain output token count (larger than max input tokens)
        max_new_tokens = 80
    )

# Convert token IDs back into text
eng_text = tokenizer.batch_decode(generated, skip_special_tokens = True)
print(eng_text)

['The sandwich was great!', 'Delivery very fast, excellent product!', "I didn't like the quality."]


### Duckdb UDF

In [10]:
import duckdb

# Initialize database path
db_file = '../olist_medallion_architecture.db'

# Connect to database file
conn = duckdb.connect(db_file, read_only = True)

In [17]:
import os
import torch
import pyarrow as pa
from dotenv import load_dotenv 
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# from duckdb.sqltypes import VARCHAR

# Get Huggingface token
load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')

# Load translation model onto GPU (once)
model_name = 'facebook/nllb-200-distilled-600M'
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang = 'por_Latn')
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name, dtype = torch.float16
).to("cuda")
model.eval()

# Define output language and batch size
ENG_TOKEN_ID = tokenizer.convert_tokens_to_ids("eng_Latn")
GPU_BATCH_SIZE = 64 

def translate_port_to_eng_arrow(text_col: pa.Array) -> pa.Array:
    # Convert Arrow array to Python list, replacing NULLs with ""
    raw_texts = [('' if val is None else str(val).strip()) for val in text_col.to_pylist()]
    
    # Deduplicate non-empty strings within the DuckDB vector chunk
    unique_non_empty = sorted({t for t in raw_texts if t}, key = len)
    cache = {'': ''}

    # Translate in batches
    for i in range(0, len(unique_non_empty), GPU_BATCH_SIZE):
        # Slice batch of text samples
        batch = unique_non_empty[i : i + GPU_BATCH_SIZE]

        # Convert input strings to token ID tensors
        inputs = tokenizer(
            batch, return_tensors = 'pt', padding = True, truncation = True
        ).to("cuda")

        # Translation loop
        with torch.no_grad():
            generated = model.generate(
                **inputs,
                forced_bos_token_id = ENG_TOKEN_ID,
                max_new_tokens = 80,
                max_length = None
            )

        # Convert token IDs back into text
        decoded = tokenizer.batch_decode(generated, skip_special_tokens = True)
        
        # Load before and after text into cache dict
        for orig, trans in zip(batch, decoded):
            cache[orig] = trans

    # Map back in original order (replace NULL's w/ empty string '')
    translated = [cache[t] for t in raw_texts]
    return pa.array(translated, type = pa.string())


# Register UDF in duckdb
try:
    conn.remove_function('translate_port_to_eng_arrow')
except Exception:
    pass

conn.create_function(
    'translate_port_to_eng_arrow', 
    translate_port_to_eng_arrow, 
    ['VARCHAR'], 
    'VARCHAR',
    type = 'arrow',
    null_handling = 'special')

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 917.59it/s] 


In [18]:
conn.query("""
    SELECT
        review_id,
        review_comment_title,
        translate_port_to_eng_arrow(review_comment_title) AS title,
        review_comment_message,
        translate_port_to_eng_arrow(review_comment_message) AS comment
    FROM bronze_reviews
    WHERE review_comment_message IS NOT NULL
    LIMIT 10;
""")

review_idVARCHAR,review_comment_titleVARCHAR,titleVARCHAR,review_comment_messageVARCHAR,commentVARCHAR
e64fb393e7b32834bb789ff8bb30750e,NULL,,Recebi bem antes do prazo estipulado.,I got it well before the deadline.
f7c4243c7fe1938f181bec41a392bdeb,NULL,,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,Congratulations Lannister shops I love shopping online safe and practical congratulations everyone happy Easter
8670d52e15e00043ae7de4c01cc2fe06,recomendo,I recommend,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,On the website the label of the device is printed as 3desinfectant and when you get this one with another name... update with the correct label since it's the same device
4b49719c8a200003f700d3d986ea1a19,NULL,,"Mas um pouco ,travando...pelo valor ta Boa.","But a little, by the way... for the good value."
3948b09f7c818e2d86c9a546758b2335,Super recomendo,I highly recommend it.,"Vendedor confiável, produto ok e entrega antes do prazo.","Trusted seller, product ok and delivery ahead of time."
9314d6f9799f5bfba510cc7bcd468c01,NULL,,"GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E ESSA COMPRA AGORA ME DECPCIONOU","I'd like to know what I've had, always received and bought now I'm not happy."
373cbeecea8286a2b66c97b1b157ec46,Não chegou meu produto,My product didn't arrive.,Péssimo,I 'm sorry .
d21bbc789670eab777d27372ab9094cc,Ótimo,That 's great .,Loja nota 10,Shop note 10
0e0190b9db53b689b285d3f3916f8441,NULL,,obrigado pela atençao amim dispensada,Thank you for your attention.
fe3db7c069d694bab50cc43463f91608,NULL,,"A compra foi realizada facilmente. A entrega foi efetuada muito antes do prazo dado. O produto já começou a ser usado e até o presente, sem problemas.","The purchase was made easily. The delivery was made well before the given deadline. The product has already begun to be used and to date, without problems."


In [20]:
# Close database connection
conn.close()